# Create color usage

`lineage_tracing/lineage` experiments are analyzed with **fishtank**, not MERlin — this notebook replaces `05_create_data_organization.ipynb` (used by every other variant/acquisition-type) with fishtank's own config file: `color_usage` (per-round/color target table) plus `decoding_strategy` (per-target decode method + reference file).

Also writes a second, single-row `color_usage_{SAMPLE_NAME}_mf.csv` for cross-modality registration against the sibling merfish acquisition's cells/DAPI round (see notebook 07).

In [ ]:
import os
import sys
from pathlib import Path

MERCI_DIR    = Path(os.getcwd()).parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/before_imaging/regular/)
SAMPLE_DIR   = MERCI_DIR.parent                  # experiment root, e.g. LT056_sample_04/lineage/
METADATA_DIR = SAMPLE_DIR / "metadata"
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.fishtank_config import create_color_usage_csv, create_decoding_strategy_csv
from MERci.acquisition.pipeline_config import load_pipeline_config
from MERci.common.experiment_info import resolve_sample_identity

# PIPELINE_ID names this variant's own pipeline.yaml -- see notebook 01's own
# comment on PIPELINE_ID/PIPELINE_CONFIG for the rationale.
PIPELINE_ID     = "lineage_tracing_lineage"   # EDIT ME -- one of the ids in pipeline_export.PIPELINES
PIPELINE_CONFIG = load_pipeline_config(MERCI_DIR / "data" / "pipelines" / PIPELINE_ID / "pipeline.yaml")

# SAMPLE_NAME is the TRUE top-level experiment id -- must match what notebooks
# 02-04 used (see notebook 02's docstring for why this isn't SAMPLE_DIR.name).
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)

print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")

## Round-tag mapping (manual)

Fishtank's `color_usage` table tags each imaging series' per-color round with a plain label (`"r1"`, `"beads"`, `"DAPI"`, `"empty"`, or blank = not imaged). This mapping is a per-protocol choice — which round images which target, in which color — and is **not** derived from `round_info.csv`/`round_bit_color_map.csv`; enter it directly below, one row per imaging series. `series` is a `{fov:03d}`-style path pattern **relative to whatever `--input` root the fishtank scripts use** (notebook 06 defaults that root to `SAMPLE_DIR/data`, so series values would look like `hybs/H01/....tif` under MERci's own layout — adjust to match your actual acquisition structure). `frames` is the frame-table CSV path for that series (see `metadata/frame-table-*.csv`, notebook 01).

In [ ]:
COLOR_USAGE_COLORS = tuple(PIPELINE_CONFIG.fishtank.color_usage_colors)

COLOR_USAGE_ROWS = [
    # {"series": "data/....tif", "frames": str(METADATA_DIR / "frame-table-....csv"),
    #  "650": "r1", "560": "r2", "488": "beads", "405": "DAPI"},
]

if not COLOR_USAGE_ROWS:
    print("WARNING: COLOR_USAGE_ROWS is empty -- fill in the round-tag mapping above.")

color_usage_path = METADATA_DIR / f"color_usage_{SAMPLE_NAME}.csv"
create_color_usage_csv(color_usage_path, COLOR_USAGE_ROWS, colors=COLOR_USAGE_COLORS)
print(f"Saved: {color_usage_path}")

## Cross-modality color usage (merfish reference round)

A single-row `color_usage` naming just the merfish acquisition's cells/DAPI round — used by `cellpose_ft_mf.slurm` (notebook 07) to segment nuclei in the merfish images for cross-modality registration, the same way `05_create_data_organization.ipynb`'s cells round is used on the merfish side.

In [3]:
COLOR_USAGE_ROWS_MF = [
    # {"series": "data/....tif", "frames": "...",
    #  "650": "", "560": "", "488": "beads", "405": "DAPI"},
]

color_usage_mf_path = METADATA_DIR / f"color_usage_{SAMPLE_NAME}_mf.csv"
create_color_usage_csv(color_usage_mf_path, COLOR_USAGE_ROWS_MF, colors=COLOR_USAGE_COLORS)
print(f"Saved: {color_usage_mf_path}")

Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\metadata\color_usage_251225_LT027_saving_time_mf.csv


## Decoding strategy

One row per decode target (e.g. the integration-barcode `intBC` target plus any base-editing targets). `reference_file`/`whitelist` point at the shared reference files notebook 07 copies into `fishtank/reference/` — set `FISHTANK_CLUSTER_DIR` to where this experiment's `fishtank/` folder will live once transferred to the cluster (matches the `data_home`-style cluster path you'd put in `experiment_info.yaml`, notebook 06).

In [ ]:
LINEAGE_LIB_VERSION  = PIPELINE_CONFIG.fishtank.lineage_lib_version   # selects MERci/data/configs/fishtank/reference/{version}/
FISHTANK_CLUSTER_DIR = f"/n/CHANGE_ME/PROJECT/experiments/{SAMPLE_NAME}/fishtank"   # edit me

# Pipeline-level decode-target menu (pipeline.yaml's fishtank.targets) --
# each target's reference_file/whitelist there is already substituted for
# LINEAGE_LIB_VERSION; joined with FISHTANK_CLUSTER_DIR (per-experiment, not
# known at pipeline.yaml-authoring time) here.
TARGETS = [
    (t.name, t.method,
     f"{FISHTANK_CLUSTER_DIR}/reference/{t.reference_file}" if t.reference_file else "",
     f"{FISHTANK_CLUSTER_DIR}/reference/{t.whitelist}" if t.whitelist else "")
    for t in PIPELINE_CONFIG.fishtank.targets
]

decoding_strategy_path = METADATA_DIR / f"decoding_strategy_{SAMPLE_NAME}.csv"
create_decoding_strategy_csv(decoding_strategy_path, TARGETS)
print(f"Saved: {decoding_strategy_path}")